In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("novagen_dataset.csv")

In [7]:
df.isnull().sum()

Age                      0
BMI                      0
Blood_Pressure           0
Cholesterol              0
Glucose_Level            0
Heart_Rate               0
Sleep_Hours              0
Exercise_Hours           0
Water_Intake             0
Stress_Level             0
Target                   0
Smoking                  0
Alcohol                  0
Diet                     0
MentalHealth             0
PhysicalActivity         0
MedicalHistory           0
Allergies                0
Diet_Type__Vegan         0
Diet_Type__Vegetarian    0
Blood_Group_AB           0
Blood_Group_B            0
Blood_Group_O            0
dtype: int64

In [8]:
df.head()

,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,...,Diet,MentalHealth,PhysicalActivity,MedicalHistory,Allergies,Diet_Type__Vegan,Diet_Type__Vegetarian,Blood_Group_AB,Blood_Group_B,Blood_Group_O
0,2.0,26.0,111.0,198.0,99.0,72.0,4.0,1.0,5.0,5.0,...,1,2,1,0,1,False,True,True,False,False
1,8.0,24.0,121.0,199.0,103.0,75.0,2.0,1.0,2.0,9.0,...,1,2,1,2,2,False,False,True,False,False
2,81.0,27.0,147.0,203.0,100.0,74.0,10.0,-0.0,5.0,1.0,...,2,0,0,1,0,True,False,False,False,False
3,25.0,21.0,150.0,199.0,102.0,70.0,7.0,3.0,3.0,3.0,...,1,2,1,2,0,True,False,False,True,False
4,24.0,26.0,146.0,202.0,99.0,76.0,10.0,2.0,5.0,1.0,...,2,0,2,0,2,False,True,False,True,False


In [13]:
X = df.drop("Target",axis=1)
y = df["Target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size =0.3 , random_state=42)

In [14]:
#Normalization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
#Model 1 - Logistic Regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    max_iter=1000
)
lr.fit(X_train_scaled,y_train)
y_pred_lr = lr.predict(X_test_scaled)

In [19]:
# Model 2 - KNN
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(
    n_neighbors = 5,
    metric = "euclidean"
)
knn.fit(X_train_scaled,y_train)
y_pred_knn = knn.predict(X_test_scaled)

In [20]:
# Model 3 - random forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)
rf.fit(X_train_scaled,y_train)
y_pred_rf = rf.predict(X_test_scaled)


In [21]:
# Model 4 - Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

gbc = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gbc.fit(X_train_scaled,y_train)
y_pred_gbc = gbc.predict(X_test_scaled)

In [25]:
#ensemble learning
from sklearn.ensemble import VotingClassifier

vr = VotingClassifier(
    estimators = [
        ("lr", lr),
        ("knn",knn),
        ("rf", rf),
        ("gbc",gbc)
    ]
)

In [26]:
vr.fit(X_train_scaled,y_train)

VotingClassifier(estimators=[('lr',
                              LogisticRegression(max_iter=1000,
                                                 solver='liblinear')),
                             ('knn', KNeighborsClassifier(metric='euclidean')),
                             ('rf',
                              RandomForestClassifier(n_estimators=200,
                                                     random_state=42)),
                             ('gbc',
                              GradientBoostingClassifier(n_estimators=150,
                                                         random_state=42))])

In [27]:
y_pred = vr.predict(X_test_scaled)

In [32]:
from sklearn.metrics import accuracy_score, classification_report
print("accuracy-final:", accuracy_score(y_pred, y_test)*100,"%")
print("accuracy-lr:", accuracy_score(y_pred_lr, y_test)*100,"%")
print("accuracy-knn:", accuracy_score(y_pred_knn, y_test)*100,"%")
print("accuracy-rf:", accuracy_score(y_pred_rf, y_test)*100,"%")
print("accuracy-gbc:", accuracy_score(y_pred_gbc, y_test)*100,"%")
print("classification report:\n",classification_report(y_pred,y_test))

accuracy-final: 92.07678883071553 %
accuracy-lr: 81.8848167539267 %
accuracy-knn: 86.59685863874346 %
accuracy-rf: 93.54275741710296 %
accuracy-gbc: 92.73996509598604 %
classification report:
               precision    recall  f1-score   support

           0       0.94      0.90      0.92      1413
           1       0.91      0.94      0.92      1452

    accuracy                           0.92      2865
   macro avg       0.92      0.92      0.92      2865
weighted avg       0.92      0.92      0.92      2865

